In [1]:
import os
import json
from typing import List, Literal, Optional, Text, Union, Dict, Any
from IPython.display import Image, display
from langchain_core.runnables.graph_mermaid import MermaidDrawMethod
from agents.agents_modules.workflow import build_agent_workflow
from agents.dataloader import load_dataset_by_name, extract_example
from agents.utilities.agent_utils import save_result_to_json
from agents.llm_model import UnifiedModel, model_name
from agents.agent_prompts import END_TO_END_GENERATION_PROMPT, input_prompt #,CONTENT_SELECTION_PROMPT

/Users/chinonsoosuji/opt/anaconda3/envs/lang2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
provider = "openai" #ollama, openai, hf, aixplain
process_flow = build_agent_workflow(provider=provider)
# display(Image(process_flow.get_graph(xray=True).draw_mermaid_png()))

In [3]:
from load_data import extract_mtriples, extract_mtriples_by_category
xml_path = "data/long-inputs_GREC_3_10.xml"
dataset = extract_mtriples(xml_path)

print(f"Number of data points: {len(dataset)}")
num = 1
data = dataset[num]

Number of data points: 533


In [4]:
query = f"""You are an agent designed to generate text from data for a data-to-text natural language generation. You can be provided data in the form of xml, table, meaning representations, graphs etc. 
Your task is to generate the appropriate text given the data information without omitting any field or adding extra information in essence called hallucination.

Dataset: 'webnlg'
Here is the data, generate text using the provided data:
{data}"""

initial_state = {
    "user_prompt": query,
    "raw_data": data,
    "history_of_steps": [],
    "final_response": "",
    "next_agent": "",
    "next_agent_payload": "",
    "current_step": 0,
    "iteration_count": 0,
    "max_iteration": 100,
}

state = process_flow.invoke(initial_state, config={"recursion_limit": initial_state["max_iteration"]})
prediction = state['final_response']



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "Final Answer",
  "action_input": [
    ["Abu_Dhabi", "country", "United_Arab_Emirates"],
    ["United_Arab_Emirates", "capital", "Abu_Dhabi"],
    ["Abu_Dhabi", "type", "Capital_city"],
    ["Abu_Dhabi", "type", "Metropolis"],
    ["Abu_Dhabi", "areaTotal", "972.0"],
    ["Abu_Dhabi", "areaTotal", "972000000.0"],
    ["Abu_Dhabi", "populationTotal", "1512000"],
    ["Abu_Dhabi", "governmentType", "Municipality"],
    ["Abu_Dhabi", "timeZone", "Time_in_the_United_Arab_Emirates"],
    ["Abu_Dhabi", "utcOffset", "+4"],

    ["Abu_Dhabi_Urban_Planning_Council", "jurisdiction", "Abu_Dhabi"],
    ["Cabinet_of_the_United_Arab_Emirates", "headquarter", "Abu_Dhabi"],
    ["Quest_Arabiya", "headquarter", "Abu_Dhabi"],
    ["Romai_Sports", "headquarter", "Abu_Dhabi"],
    ["Caracal_International", "locationCity", "Abu_Dhabi"],
    ["Emirates_Defence_Industries_Company", "locationCity", "Abu_Dhabi"],
    ["Adcom_Systems", "location

In [5]:
save_result_to_json(state, filename=f"webnlg_long_{num}.json")

[SAVED] Agent result saved to: results/webnlg_long_1.json


In [6]:
print(prediction)

Abu Dhabi is the capital city and a major metropolis of the United Arab Emirates. It serves as the capital of the country and is recognized for its significant size and population, covering an area of 972.0 square kilometers (or 972,000,000.0 square meters) and home to a population of 1,512,000 people. The city operates as a municipality and is located in the time zone known as Time in the United Arab Emirates, with a UTC offset of +4.

Abu Dhabi is the jurisdiction of the Abu Dhabi Urban Planning Council and hosts the headquarters of several organizations, including the Cabinet of the United Arab Emirates, Quest Arabiya, and Romai Sports. The city is also the location for companies such as Caracal International, Emirates Defence Industries Company, Adcom Systems, and Waha Capital. Aldar Properties, Fakhro Group, and DeSimone Consulting Engineers all serve the Abu Dhabi region, while Cleveland Clinic Abu Dhabi and Sheikh Khalifa Medical City are also situated in the city. Educational i